# I5. RFI and flagging

The fifth notebook of the introductory track ({doc}`../index`). Radio
frequency interference (RFI) is any signal that is not from the sky: phones,
navigation satellites, aircraft. This notebook looks at real RFI in the
fixture, then flags A1 small, whose RFI is simulated with a known mask, so
every flag can be scored. About a minute.

**Learning goals.** After this notebook you can

- recognise RFI in time, frequency and baseline length;
- say what MeerKAT's static mask is and what it costs;
- run `rang-core flag` and score a flagger against a truth mask: recall,
  false flags, data lost;
- explain why a flag is a decision with a cost: flagged data are lost,
  unflagged RFI biases what remains;
- name what this project tested about flagging and what came out null.

In [ ]:
%matplotlib inline
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from rang import core_ms
from rang import simtruth as st
from rang.notebook import data, flags

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "research" / "results").is_dir()), None)
WORK = data.default_cache_dir() / "intro-i5"
shutil.rmtree(WORK, ignore_errors=True)
WORK.mkdir(parents=True)
ds = data.load_fixture(data.find_fixture() or data.download_fixture(data.default_cache_dir()))
s = ds.state

## 1. RFI in real data

The fixture's FLAG column holds the flags of MeerKAT's online flagger. By
channel:

In [ ]:
display(flags.flag_fraction_by_channel(ds))

A1's RFI model names the transmitters it simulates and the A1 channels they
fall in. The same frequency ranges in the fixture's 4,096 channels:

In [ ]:
d = st.design("small")
events = st.rfi_events(d, 104)
step = (d.fmax - d.fmin) / d.nchan
freq = s["frequencies"]
cross = s["antenna1"] != s["antenna2"]
flagged = s["original_flags"][cross].any(-1).mean(axis=0)  # per channel, over cross rows
named = {}
for e in events:
    if e["kind"] != "burst":
        lo, hi = d.fmin + min(e["chans"]) * step, d.fmin + (max(e["chans"]) + 1) * step
        named[e["name"]] = (lo, hi)
inside = np.zeros(len(freq), bool)
rows = []
for name, (lo, hi) in named.items():
    sel = (freq >= lo) & (freq < hi)
    inside |= sel
    rows.append({"transmitter (A1's model)": name, "MHz": f"{lo / 1e6:.0f}-{hi / 1e6:.0f}", "fixture flagged": f"{flagged[sel].mean():.0%}"})
rows.append({"transmitter (A1's model)": "everything else", "MHz": "", "fixture flagged": f"{flagged[~inside].mean():.0%}"})
pd.DataFrame(rows)

RFI is mostly narrow in frequency and persistent (satellites, GSM) or
intermittent (aircraft transponders), with occasional broadband bursts. A
transmitter on the horizon is far from the phase centre, so its fringe
rotates fast and averages down on long baselines: RFI is strongest on short
baselines.

## 2. The static mask

Channels that are always occupied are flagged in advance with a **static
mask**. rang ships MeerKAT's L-band mask from Tricolour:

In [ ]:
mask_file = Path(st.ITRF).parent / "4k_lband_meerkat.staticmask"
table = np.load(mask_file)
static, chans = table["mask"][0], table["chans"][1]
print(f"{mask_file.name}: {static.size} channels of {(chans[1] - chans[0]) / 1e3:.1f} kHz, {static.mean():.1%} masked")
in_static = np.interp(freq, chans, static.astype(float)) > 0.5
print(f"fixture: {in_static.mean():.1%} of its channels fall in the mask; "
      f"flagged fraction there {flagged[in_static].mean():.0%}, elsewhere {flagged[~in_static].mean():.0%}")

On A1 small's 64 channels of 13.4 MHz, a channel counts as masked when the
mask covers any part of it, so applying the mask on every baseline throws
away far more than 41 % (section 4).

## 3. Simulated RFI with a known mask

Write A1 small and its noiseless twin (I4). Next to the MS lies
`a1-small.rfi_mask.npz`: every cell that received RFI (`any`) and every
cell where it exceeds 5σ (`above_5sigma`).

In [ ]:
for extra in ([], ["--noiseless", "--name", "a1-small-twin"]):
    subprocess.run([sys.executable, "-m", "rang.simtruth_ms", "generate", "small", str(WORK), "--workers", "2", *extra],
                   capture_output=True, check=True, env={**os.environ, "RAYON_NUM_THREADS": "2"})
MS = WORK / "a1-small.ms"
npz = np.load(WORK / "a1-small.rfi_mask.npz")
truth = {k: np.unpackbits(npz[k], axis=1, count=d.nchan).astype(bool) for k in ("any", "above_5sigma")}
for k, m in truth.items():
    print(f"truth mask {k}: {m.mean():.1%} of cells")

with core_ms.open_table(str(MS)) as t:
    a1, a2, uvw = t.getcol("ANTENNA1"), t.getcol("ANTENNA2"), t.getcol("UVW")
    noisy = t.getcol("DATA")
with core_ms.open_table(str(WORK / "a1-small-twin.ms")) as t:
    twin = t.getcol("DATA")
length = np.linalg.norm(uvw, axis=1)
print("fraction of cells hit by RFI, by baseline length:")
for lo, hi in ((0, 500), (500, 1000), (1000, 3000), (3000, 8000)):
    sel = (length >= lo) & (length < hi)
    print(f"  {lo}-{hi} m: {truth['any'][sel].mean():.1%}")

In [ ]:
nbl = 120
fig, axes = plt.subplots(1, 2, figsize=(11, 3.4), sharey=True)
for ax, (p, q) in zip(axes, ((0, 1), (0, 14))):
    rows_pq = np.flatnonzero((a1 == p) & (a2 == q))
    rfi = np.abs(noisy[rows_pq, :, 0] - twin[rows_pq, :, 0])
    im = ax.imshow(np.log10(rfi + 1e-3), aspect="auto", origin="lower", cmap="viridis", vmin=-2.5, vmax=0.5,
                   extent=[d.fmin / 1e6, d.fmax / 1e6, 0, len(rows_pq)])
    ax.set(title=f"m{4 * p:03d}-m{4 * q:03d}, {length[rows_pq].mean():.0f} m: log10 |DATA − twin| (XX)", xlabel="frequency (MHz)")
axes[0].set_ylabel("integration")
fig.colorbar(im, ax=axes, label="log10 Jy")
plt.show()

DATA minus the twin is noise plus RFI. On the short baseline the
transmitters stand out; on the long one most have decorrelated into the
noise.

## 4. Flagging and scoring

`rang-core flag CONFIG MS` runs a Tricolour strategy (rang's Rust port) on
DATA and writes FLAG, saving the old flags first as a CASA flag version. Two
strategies: E4's tutorial strategy (the static mask on baselines up to 1 km,
then SumThreshold at 5σ), and SumThreshold alone.

In [ ]:
src = ROOT / "examples" / "ms-to-image" if ROOT else None
config = (src / "config.yaml").read_text(encoding="utf-8")
tutorial = (src / "strategy.yaml").read_text(encoding="utf-8")
start = tutorial.index("  - name: static_mask")
end = tutorial.index("  - name: background_flags")
strategies = {"tutorial (static mask up to 1 km + SumThreshold 5σ)": tutorial, "SumThreshold 5σ only": tutorial[:start] + tutorial[end:]}


def flag_with(strategy_text, name):
    run = WORK / name
    run.mkdir()
    shutil.copytree(MS, run / "a1-small.ms")
    (run / "strategy.yaml").write_text(strategy_text, encoding="utf-8")
    (run / "config.yaml").write_text(config, encoding="utf-8")
    p = subprocess.run([sys.executable, "-m", "rang.core_cli", "flag", "config.yaml", "a1-small.ms"], cwd=run,
                       capture_output=True, text=True, env={**os.environ, "RAYON_NUM_THREADS": "2"})
    assert p.returncode == 0, p.stdout + p.stderr
    with core_ms.open_table(str(run / "a1-small.ms")) as t:
        return t.getcol("FLAG").any(-1)


results = {label: flag_with(text, f"run{i}") for i, (label, text) in enumerate(strategies.items())}
results["no flags"] = np.zeros_like(truth["any"])
results["the truth mask (an oracle)"] = truth["any"]

Scores: **recall**, the share of RFI cells flagged; **false flags**, the
share of clean cells flagged; **lost**, all flagged cells. And the cost of
what stays: the rms of DATA − twin on unflagged XX and YY cells in units of
σ. Pure noise gives 1; anything above is unflagged RFI.

In [ ]:
excess = np.abs(noisy - twin)[..., [0, 3]]
pd.DataFrame(
    [
        {
            "flags": label,
            "lost": f"{f.mean():.1%}",
            "recall (any)": f"{f[truth['any']].mean():.1%}",
            "recall (>5σ)": f"{f[truth['above_5sigma']].mean():.1%}",
            "false flags": f"{f[~truth['any']].mean():.1%}",
            "rms of what stays / σ": f"{np.sqrt(np.mean(excess[~f] ** 2) / 2) / d.sigma:.2f}",
        }
        for label, f in results.items()
    ]
)

SumThreshold at 5σ recalls about as much RFI as it falsely flags clean
cells, so on A1 small it is no better than chance. It looks for outliers
against a smooth background; a likely reason, not tested here, is that the
simulated transmitters fill whole 13.4 MHz channels in most integrations and
so look like background. The static mask finds them because it knows the
frequencies in advance, at the price of clean data in the same channels. The
fixture's channels are 64 times narrower; notebook 02 steps SumThreshold
through them.

E4 scored more strategies on the same data
({gh}`research/results/e4-tutorial/flag_variants.json`); its tutorial row
is the run above, and the numbers agree:

In [ ]:
if ROOT:
    archive = json.loads((ROOT / "research" / "results" / "e4-tutorial" / "flag_variants.json").read_text(encoding="utf-8"))
    display(pd.DataFrame(
        [{"strategy": k, "flagged": f"{v['flagged']:.1%}", "recall (any)": f"{v['any']['recall']:.1%}", "false flags": f"{v['any']['false']:.1%}"} for k, v in archive.items()]
    ))
else:
    print("skipped: the archive is in a source checkout")

## 5. What this project found about flagging

rang's flagging research asked whether flags chosen with the help of the
calibration solution (gain total variation, TV) beat the standard
CARACal/Tricolour first pass. It did not:

- the TV-guided threshold policy was closed as **null** by its frozen
  decision rule ({gh}`research/flagging-verdict.md`; notebooks 05 and 06);
- a flagger trained on gain TV stopped at its go/no-go test, before any
  training ({gh}`research/f1-tv-flagger.md`);
- the follow-up flaggers F2 and F3 failed their own pre-registered tests
  ({gh}`research/f2-flagger.md`, {gh}`research/f3-flagger.md`).

The records keep the failed arms as carefully as a positive result would be
kept.

## Exercises

1. Make a strategy with the static mask on every baseline (remove the
   `uvrange` line). Score it. How much data does it cost, and what does the
   rms column gain?
2. Lower SumThreshold's `outlier_nsigma` to 3. Recall and false flags?
3. Compute recall separately for the `persistent`, `intermittent` and
   `burst` events (each event's `chans` and `amp_t` in `truth.json`).
4. In the fixture, which baselines carry the most flags? Use
   `flags.flag_fraction_by_antenna(ds)`.

## Where next

- {doc}`i6-calibration`: calibrating A1 small once it is flagged.
- Notebook {doc}`02-flagging-step-by-step` runs SumThreshold step by step on
  the fixture; {doc}`05-the-flagging-experiment` is the experiment and its
  null; {doc}`../../concepts/flagging`.
- {doc}`00-ms-to-image` runs the same flag stage inside `rang-core run`.